# Quantum-number transitions

:::{autolink-concat}
:::

The workflows described in {doc}`/usage/reaction` match every intermediate edge of the solved transitions to particles from a database. If you are only interested in which quantum numbers are allowed — for instance, which $J^{PC}$ resonances can appear in a Dalitz-plot decomposition — that matching step is unnecessary. This page shows how to generate transitions directly at the $J^{P(C)}$ level with the {mod}`.workflow` module: the intermediate states then remain *sets* of quantum numbers and are never matched against a particle database.

In [ ]:
import graphviz

import qrules.io
from qrules.quantum_numbers import EdgeQuantumNumbers
from qrules.workflow import (
    create_qn_problem_sets,
    find_qn_transitions,
    generate_qn_transitions,
)

PDG = qrules.load_pdg()

## Quantum-number problem sets

As an example, take the reaction $J/\psi \to \gamma\pi^0\pi^0$ with two $f_0$ resonances as allowed intermediate states. {func}`.create_qn_problem_sets` fans the initial and final state out into a {class}`.QNProblemSet` for every decay topology, kinematic permutation, and combination of allowed interaction types:

In [ ]:
qn_problem_sets = create_qn_problem_sets(
    initial_state=["J/psi(1S)"],
    final_state=["gamma", "pi0", "pi0"],
    particle_db=PDG,
    allowed_intermediate_particles=["f(0)(980)", "f(0)(1500)"],
)
sum(len(problems) for problems in qn_problem_sets.problem_sets.values())

## Solve at the quantum-number level

{func}`.find_qn_transitions` solves these problem sets purely at the quantum-number level: no particle database is consulted for the intermediate states. It returns {obj}`.QNTransition`s, whose states and interactions are property maps of quantum numbers instead of {class}`.Particle` and {class}`.InteractionProperties` objects.

In [ ]:
qn_transitions = find_qn_transitions(qn_problem_sets)
len(qn_transitions)

The transitions differ only in the quantum numbers of the intermediate state and the $LS$-couplings of the interaction nodes. They can be visualized with {func}`.asdot`, just like ordinary transitions:

In [ ]:
dot = qrules.io.asdot(qn_transitions[0], render_node=True)
graphviz.Source(dot)

Since the property maps can be inspected directly, it is easy to summarize for example the allowed $J^{PC}$ quantum numbers of the intermediate state:

In [ ]:
{
    (
        state[EdgeQuantumNumbers.spin_magnitude],
        state[EdgeQuantumNumbers.parity],
        state[EdgeQuantumNumbers.c_parity],
    )
    for transition in qn_transitions
    for state in transition.intermediate_states.values()
}

## Reaction-level interface

The staged workflow above offers full control over the problem sets, but the default use-case is covered by a single call to {func}`.generate_qn_transitions`, the quantum-number-level counterpart of {func}`.generate_transitions`. It returns a {class}`.QNReactionInfo`, which resolves the initial and final states to {class}`.Particle` instances — they are fully determined by their PID — while the intermediate states remain quantum-number property maps:

In [ ]:
reaction = generate_qn_transitions(
    initial_state="J/psi(1S)",
    final_state=["gamma", "pi0", "pi0"],
    particle_db=PDG,
    allowed_intermediate_particles=["f(0)(980)", "f(0)(1500)"],
)
{i: particle.name for i, particle in reaction.final_state.items()}

The rendered transitions now label the initial and final states by particle name:

In [ ]:
dot = qrules.io.asdot(reaction.transitions[0], render_node=True)
graphviz.Source(dot)

## Many-body reactions

Since the intermediate states never have to be matched against a particle database, solving at the quantum-number level keeps reactions with four- and even five-body final states feasible (see [ComPWA/qrules#27](https://github.com/ComPWA/qrules/issues/27)). Take $J/\psi \to K^+K^-\pi^+\pi^-$ with $\phi(1020)$ and $\rho(770)$ resonances, where {func}`.create_qn_problem_sets`' :code:`final_state_groupings` argument limits the subsystems to $\phi\to K^+K^-$ and $\rho^0\to\pi^+\pi^-$:

In [ ]:
%%time
reaction_4body = generate_qn_transitions(
    initial_state="J/psi(1S)",
    final_state=["K+", "K-", "pi+", "pi-"],
    particle_db=PDG,
    allowed_intermediate_particles=["phi(1020)", "rho(770)"],
    allowed_interaction_types="strong",
    final_state_groupings=[["K+", "K-"], ["pi+", "pi-"]],
)
len(reaction_4body.transitions)

The transitions can be summarized by collapsing them per decay topology with {func}`.asdot`'s :code:`collapse_graphs` argument, where the intermediate edges then list the allowed states in $I^G(J^{PC})$ notation:

In [ ]:
graphviz.Source(qrules.io.asdot(reaction_4body, collapse_graphs=True))

and the intermediate states are limited to the following charges, isospins, spins, and parities:

In [ ]:
{
    (
        state[EdgeQuantumNumbers.charge],
        state[EdgeQuantumNumbers.isospin_magnitude],
        state[EdgeQuantumNumbers.spin_magnitude],
        state[EdgeQuantumNumbers.parity],
    )
    for transition in reaction_4body.transitions
    for state in transition.intermediate_states.values()
}

Even a five-body final state remains tractable — here with an additional $\pi^0$, which lets the number of allowed decay topologies and quantum-number combinations grow substantially:

In [ ]:
%%time
reaction_5body = generate_qn_transitions(
    initial_state="J/psi(1S)",
    final_state=["K+", "K-", "pi+", "pi-", "pi0"],
    particle_db=PDG,
    allowed_intermediate_particles=["phi(1020)", "rho(770)"],
    allowed_interaction_types="strong",
    final_state_groupings=[["K+", "K-"], ["pi+", "pi-"]],
)
f"{len(reaction_5body.transitions):,} transitions over {len(reaction_5body.group_by_topology())} topologies"

:::{seealso}
To generate particle-level transitions — with the intermediate states matched to a particle database, as required for an amplitude model — use {func}`.find_solutions` or {func}`.generate_transitions` as described in {doc}`/usage/reaction`. The problem sets can be extended with additional quantum numbers and rules before solving, as demonstrated in {doc}`/usage/spin-projections`. Reactions with two initial states are described in {doc}`/usage/production`.
:::